# SANKHYA walkthrough: model to verified answer to what-if (#537)

This notebook drives the Python API the way a planner would: read a model, solve it under
two independent engines, have a script that shares no code with the solver check the
answer, then ask three what-if questions SANKHYA answers directly - a parametric sweep, an
infeasibility repair, and a warm-started re-solve after an edit.

**Scope, stated up front.** #517 (a dedicated refinery case study) does not exist in this
repository yet, so `demo/crude_blend.mps` stands in below - a small MAX-sense crude-blending
LP (Arab Light / Bonny Light / Murban into a diesel pool, throughput and sulphur
constrained) already used the same way in `docs/BENCHMARKS.md` for #522 and #524. Every
number below is computed by this notebook when it runs, not typed in; the one exception -
the 52-week rolling-horizon study - is too slow for a CI notebook to re-run every PR, so
that section cites the committed CSV it was measured into instead of repeating the run.

This notebook is executed end to end on every pull request (`.github/workflows/ci.yml`); a
cell that raises fails the build, which is what keeps it from rotting.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "bindings" / "python"))
sys.path.insert(0, str(REPO_ROOT / "tools"))

import sankhya

MODEL = REPO_ROOT / "demo" / "crude_blend.mps"
print(sankhya.version())
print(MODEL, "exists:", MODEL.is_file())


## 1. Solve under two engines, then verify with a script that shares no code with the solver

`Model.solve(options=...)` runs one of SANKHYA's engines; here the revised primal simplex
and the restarted first-order method (PDHG) solve the *same* model independently. Both
answers are then written to a `.sol` file with the CLI and checked by
`tools/verify_solution.py` - a from-scratch reader and recomputation of every number
(activities, the objective, reduced costs, the dual objective), so agreement is not assumed.


In [ ]:
import subprocess
import tempfile

def cli_path():
    return sankhya.locate_executable()

def dll_env():
    # Windows only: the CLI needs MSYS2's runtime DLLs (libgomp etc.) on PATH when it is
    # not already there. A no-op on Linux, where CI runs this notebook.
    import os
    env = dict(os.environ)
    dll_dir = os.environ.get("SANKHYA_DLL_DIR")
    if dll_dir and Path(dll_dir).is_dir():
        env["PATH"] = dll_dir + os.pathsep + env.get("PATH", "")
    return env

results = {}
with tempfile.TemporaryDirectory() as tmp:
    for algorithm in ("simplex", "pdhg"):
        sol_path = Path(tmp) / f"{algorithm}.sol"
        proc = subprocess.run(
            [str(cli_path()), "solve", str(MODEL), "--option", f"algorithm={algorithm}",
             "--write-sol", str(sol_path)],
            capture_output=True, text=True, env=dll_env(), check=True)
        verify = subprocess.run(
            [sys.executable, str(REPO_ROOT / "tools" / "verify_solution.py"),
             str(MODEL), str(sol_path), "--quiet"],
            capture_output=True, text=True, check=False)
        results[algorithm] = {
            "solve_stdout": proc.stdout,
            "verify_returncode": verify.returncode,
            "verify_stdout": verify.stdout,
        }
        assert verify.returncode == 0, (
            f"{algorithm}: the independent verifier rejected this solution:\n{verify.stdout}")
        print(f"=== {algorithm} ===")
        print(proc.stdout)
        print(verify.stdout.strip())
        print()

objectives = {
    algorithm: float([line for line in r["solve_stdout"].splitlines()
                     if line.strip().startswith("objective")][0].split()[-1])
    for algorithm, r in results.items()
}
print("objectives agree to 1e-6:",
     abs(objectives["simplex"] - objectives["pdhg"]) <= 1e-6 * max(1.0, abs(objectives["simplex"])))
assert abs(objectives["simplex"] - objectives["pdhg"]) <= 1e-6 * max(1.0, abs(objectives["simplex"]))


## 2. Parametric LP: the optimal value as a function of one cost (#522)

`tools/parametric.py` walks Arab Light's per-barrel margin across a range, jumping between
breakpoints using sensitivity ranging (#220) and warm-starting (#218) from the basis that
was optimal at the point being left. Every reported point is its own fresh, independently
re-solved optimum - see the module docstring for exactly how this differs from a dedicated
parametric simplex.


In [ ]:
from parametric import sweep_cost

curve = sweep_cost(MODEL, "AL", 0.0, 5.0, {})
for row in curve:
    print(f"AL margin {row['parameter']:>8.4f}  ->  objective {row['objective']:>10.4f}  "
         f"{row['status']:<8}  {row['change']}")

assert len(curve) >= 2, "expected at least one interior breakpoint over this wide a range"
assert all(row["status"] == "optimal" for row in curve)


## 3. Infeasibility repair: the smallest relaxation that restores feasibility (#523)

`demo/crude_blend_infeasible.mps` is `crude_blend.mps` with the diesel-pool row's lower
bound raised past what the throughput and sulphur limits allow. `tools/repair_infeasibility.py`
solves the elastic relaxation (Chinneck 2008, ch. 8): minimize the weighted movement of
whichever rows have to give, phase 1; then re-optimize the real objective subject to that
movement as a ceiling, phase 2.


In [ ]:
from repair_infeasibility import repair
from verify_solution_mps import parse_mps

infeasible_source = parse_mps(REPO_ROOT / "demo" / "crude_blend_infeasible.mps")
report = repair(infeasible_source, include_bounds=False, weights={}, optimize=True, log=False)

assert report["repairable"], report.get("message")
print("phase 1:", report["phase1_status"], "- relaxations found:")
for r in report["relaxations"]:
    print(f"  {r['kind']} {r['name']} ({r['side']}) moved by {r['amount']:.4f}")
print("phase 2:", report["phase2_status"], "- objective", report["phase2_objective"])

assert report["phase1_status"] == "optimal"
assert report["phase2_status"] == "optimal"


## 4. Warm-started re-solve after an edit (#524)

`Model.solve(start=<previous result>)` resumes from the prior optimal basis instead of the
slack basis. On a model this small the effect mostly rounds a handful of pivots away to
zero; the shape of the effect - not its size at toy scale - is what matters here, so the
cell below also reports iteration counts, not just wall time.


In [ ]:
import random

warm_model = sankhya.Model.read(str(MODEL))
rng = random.Random(3)
warm_previous = None
cold_iterations = []
warm_iterations = []
for step in range(6):
    factor = 1.0 + rng.uniform(-0.10, 0.10)
    warm_model.set_cost(0, 2.40 * factor)  # AL's margin, perturbed +/-10%
    cold = warm_model.solve(log_to_console=False)
    warm = warm_model.solve(log_to_console=False, start=warm_previous)
    warm_previous = warm
    cold_iterations.append(cold.iterations)
    warm_iterations.append(warm.iterations)
    assert cold.status == warm.status == "optimal"
    assert abs(cold.objective - warm.objective) <= 1e-6 * max(1.0, abs(cold.objective))
    print(f"step {step}: cold {cold.iterations} pivots, warm {warm.iterations} pivots, "
         f"objective {warm.objective:.4f}")

print()
print(f"total: cold {sum(cold_iterations)} pivots, warm {sum(warm_iterations)} pivots")


### At refinery scale, not toy scale

The same mechanism, exercised on a 52-period refinery plan (4,628 rows, one crude's
purchase price perturbed +/-10% a week at a time,
`bench/runners/rolling_warm_start.py`) and committed as
`bench/results/rolling-warm-start-c8b9437.csv`:
warm takes **0.121x the iterations and 0.139x the wall time** of cold, summed over all 52
weeks, and the warm and cold objectives agree on all 52 weeks. That run is a 52-week solve
sequence (several minutes) and is not re-run by this notebook; the cell below only checks
that the CSV this cites is the one actually committed to the repository, so the citation
itself cannot rot even though the run isn't repeated here.


In [ ]:
csv_path = REPO_ROOT / "bench" / "results" / "rolling-warm-start-c8b9437.csv"
assert csv_path.is_file(), f"the CSV this section cites is missing: {csv_path}"

import csv as csv_module
with open(csv_path, newline="", encoding="utf-8") as handle:
    rows = list(csv_module.DictReader(handle))

total_cold = sum(int(r["cold_iterations"]) for r in rows)
total_warm = sum(int(r["warm_iterations"]) for r in rows)
agree = sum(1 for r in rows if r["answers_agree"] == "True")
print(f"{len(rows)} weeks, commit {rows[0]['git_commit']}: "
     f"cold {total_cold} iterations, warm {total_warm} ({total_warm / total_cold:.3f}x); "
     f"answers agree on {agree}/{len(rows)} weeks")

assert agree == len(rows), "the committed CSV no longer shows full agreement"


## What this notebook does not show

- **The full refinery case (#517)** - not built yet; `demo/crude_blend.mps` stands in
  throughout, as noted at the top.
- **MINLP, decomposition (#528, #525)** - not implemented.
- **Exact rational output (#521)** - not implemented; every number above is double
  precision, independently re-verified, not independently re-derived in exact arithmetic.
- **The 52-week rolling-horizon run itself** - too slow to repeat on every PR; this
  notebook checks the committed CSV exists and still agrees on every week instead of
  re-running the solve sequence.
